In [2]:
import numpy as np
import matplotlib.pyplot as plt
import math

In [3]:
#####################################################################################################

class CosmicWatchMeasurement:
    field_idx = {
        3 : 'times',
        6 : 'energies',
        7 : 'temperatures',
        8 : 'pressures',
        9 : 'dead_times',
        10 : 'coincident'
    }

    def __init__(self,
                 data_file,
                 only_coincident=True,
                 energy_cut=None):
        
        self._read_data(data_file)
        self.temperature_avg, self.temperature_std, self.temperature_max, self.temperature_min = self._get_average_param('temperatures')
        self.pressure_avg, self.pressure_std, self.pressure_max, self.pressure_min = self._get_average_param('pressures')

        if only_coincident:
            self._filter_coincident()
        if energy_cut is not None:
            self._filter_energy(energy_cut)

        self._get_rate()

    def _read_data(self,data_file):
        data = {
            field : [] for idx,field in self.field_idx.items()
        }
        with open(data_file, 'r') as f:
            for line in f.readlines():
            
                if line.startswith('#'):
                    continue
                line_data = line.split()
                for idx, field in self.field_idx.items():
                    data[field].append(float(line_data[idx]))

        for field in data:
            setattr(self,field,np.array(data[field]))

        self.total_time = self.times[-1]*1e-3

        self.dead_times = np.cumsum(self.dead_times)*1e-6
        self.times = self.times*1e-3 - self.dead_times

        self.live_time = self.times[-1]

    def _filter_coincident(self):
        mask = self.coincident == 1
        for field in self.field_idx.values():
            setattr(self,field,getattr(self,field)[mask])

    def _filter_energy(self, energy_cut):
        mask = self.energies > energy_cut
        for field in self.field_idx.values():
            setattr(self,field,getattr(self,field)[mask])

    def _get_rate(self):
        self.counts = self.times.size
        interval = self.times[-1] - self.times[0]

        self.rate = (self.counts-1) / interval * 1e3
        self.rate_err = np.sqrt(self.counts-1) / interval * 1e3

        self.rate_sys_err = (5 * self.total_time/3600/24) / interval * 1e3

    def _get_average_param(self,param):
        param_avg = np.mean(getattr(self, param))
        param_std = np.std(getattr(self, param))

        param_max = np.max(getattr(self, param))
        param_min = np.min(getattr(self, param))

        return param_avg, param_std, param_max, param_min

In [5]:
data = CosmicWatchMeasurement(data_file="angular_dist\\FileC094.txt",only_coincident=False)

In [ ]:
N = 4
times = data.times/1e-3

triggers = times[:N]
bin_seq = []
for point in triggers:
    bin_seq.append(round(point)%2)
    str_list = [str(num) for num in bin_seq]
    bin_num = int(''.join(str_list))
bin_num

['0']
['0', '1']
['0', '1', '0']
['0', '1', '0', '0']


100

In [ ]:
num = 0
for ind,flip in enumerate(bin_seq):
    num += flip*2**(N-1-ind)
num

15